# Clock Recognition and Manipulation — Full Pipeline

End-to-end deep-learning demo: read time from a **digital clock** image, then re-render an **analog clock** to show that time using two generative approaches.

| Stage | Input | Output |
|---|---|---|
| 1. Digital recognition | Digital clock photo | Recognised time `HH:MM` |
| 2. Sketch-cGAN          | Analog clock + target sketch | Re-rendered clock face |
| 3. Inpainting GAN       | Analog clock + hand masks   | Hands removed → re-composed at target time |

All heavy lifting is in [`pipeline_runners.py`](./pipeline_runners.py) — this notebook is just the orchestration.

## Setup

In [ ]:
import sys; sys.path.append('./')

import torch
from IPython.display import Image as IPImage, display

import pipeline_runners as pr
from analog_clock.analog_sketch_creator import draw_analog_clock
from analog_clock.pipeline_utils import generate_inpainting_gif, generate_sketch_cgan_gif

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Inputs
DIGITAL_CLOCK_IMAGE = './sample_inputs/digital_clock_image.png'
ANALOG_CLOCK_IMAGE  = './sample_inputs/analog_clock_image.jpeg'

# Digital pipeline weights
YOLO_DIGITS_PATH = './digital_clock/yolo_detect_hh_mm/yolo.pt'
SVHN_CNN_PATH    = './digital_clock/svhn_digit_recognition_cnn/svhn_cnn.pth'

# Analog V1 (sketch-cGAN) weights
YOLO_CLOCK_PATH      = './analog_clock/yolo_detect_clock/analog_clock_yolo_model.pt'
SKETCH_GENERATOR_PATH = './analog_clock/GAN/sketch/generator_100.pth'

# Analog V2 (inpainting) weights
YOLO_HANDS_PATH    = './analog_clock/yolo_detetct_hands/yolo_hands_seconds.pt'
INPAINTING_PATH    = './analog_clock/GAN/inpainting/inpaint_gen_100.pth'
TIME_CNN_PATH      = './analog_clock/time_recognition_cnn/clock_hand_cnn_best.pth'

# Animation outputs
SKETCH_GIF_PATH     = './output_sketch.gif'
INPAINTING_GIF_PATH = './output_inpainting.gif'

---

## Stage 1 — Digital Time Recognition

**Pipeline:** YOLOv8 localises the `HH` and `MM` digit regions, then a 7-conv-layer CNN with 5 output heads (SVHN-style) predicts each digit independently.

In [ ]:
yolo_digits, svhn_cnn = pr.load_digital_models(YOLO_DIGITS_PATH, SVHN_CNN_PATH, device)

In [ ]:
digital = pr.recognize_digital_time(DIGITAL_CLOCK_IMAGE, yolo_digits, svhn_cnn, device)
target_hh, target_mm = digital.hh, digital.mm
print(f"Target time recognised: {target_hh}:{target_mm:02d}")

In [ ]:
pr.show_digital(digital)

---

## Stage 2 — Load Analog Pipelines

Both analog approaches need their generators plus a YOLO segmenter. We load everything once, then read the current time from the analog clock to use as the animation start point.

In [ ]:
yolo_clock, sketch_gen, sketch_tf = pr.load_sketch_models(
    YOLO_CLOCK_PATH, SKETCH_GENERATOR_PATH, device,
)
inpaint_models = pr.load_inpainting_models(
    YOLO_HANDS_PATH, INPAINTING_PATH, TIME_CNN_PATH, device,
)

In [ ]:
original_hh, original_mm = pr.recognize_analog_time(ANALOG_CLOCK_IMAGE, inpaint_models, device)
print(f"Analog clock currently shows: {original_hh}:{original_mm:02d}  →  target: {target_hh}:{target_mm:02d}")

---

## Stage 3 — Analog V1: Sketch-cGAN

**Pipeline:** YOLO crops the analog clock face, then a Pix2Pix-style 8-layer U-Net (`in_channels=6`: original + target sketch) re-renders the clock to display the target time.

In [ ]:
sketch_res = pr.run_sketch_cgan(
    ANALOG_CLOCK_IMAGE, target_hh, target_mm,
    yolo_clock, sketch_gen, sketch_tf, device,
)

In [ ]:
pr.show_sketch(sketch_res)

### Animated sweep

In [ ]:
gif_path = generate_sketch_cgan_gif(
    original_img_pil=sketch_res.original,
    crop_coords=sketch_res.crop_coords,
    generator=sketch_gen,
    transform=sketch_tf,
    start_hh=original_hh, start_mm=original_mm,
    target_hh=target_hh,  target_mm=target_mm,
    draw_analog_clock_fn=draw_analog_clock,
    device=device,
    step_minutes=2,
    output_path=SKETCH_GIF_PATH,
    duration_ms=200,
)
display(IPImage(filename=gif_path))

---

## Stage 4 — Analog V2: Mask-Guided Inpainting

**Pipeline:** YOLO-Seg masks each clock hand → an inpainting U-Net erases the hands from the face → each hand is rotated to the target angle and algorithmically blended back in.

In [ ]:
inpaint_res = pr.run_inpainting(ANALOG_CLOCK_IMAGE, target_hh, target_mm, inpaint_models, device)

In [ ]:
pr.show_inpainting(inpaint_res)

### Animated sweep

In [ ]:
gif_path = generate_inpainting_gif(
    img_cv=inpaint_res.img_cv,
    clean_bg=inpaint_res.clean_bg,
    center=inpaint_res.center,
    mask_h=inpaint_res.best_mask_h,
    mask_m=inpaint_res.best_mask_m,
    current_angle_h=inpaint_res.angle_h_orig,
    current_angle_m=inpaint_res.angle_m_orig,
    start_hh=original_hh, start_mm=original_mm,
    target_hh=target_hh,  target_mm=target_mm,
    step_minutes=2,
    output_path=INPAINTING_GIF_PATH,
    duration_ms=200,
    feather_radius=5,
)
display(IPImage(filename=gif_path))